# Practical PyTorch · Part 12 — Companion Notebook (Capstone)

This goes with **"From Notebook to App: Wrap a Model in a UI Anyone Can Use"**. Run the cells top to bottom (Shift+Enter).

Turn the GPU on before the image-generation section: **Runtime → Change runtime type → Hardware accelerator → GPU**.

## 0. Install what we need

Colab has PyTorch and `transformers` already, but we'll add `gradio` (for the UI) and make sure `transformers` is current. This takes a few seconds.

In [ ]:
!pip install -q gradio transformers

## 1. A model is just a function

Input in, output out — the same shape as everything else in this series. We'll load an image classifier with `pipeline()` and wrap it in a plain function that returns `{label: score}`.

In [ ]:
from transformers import pipeline

classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

def classify(img):
    results = classifier(img)
    # Turn the list of {label, score} into the {label: score} a UI wants.
    return {r["label"]: r["score"] for r in results}

## 2. Put a UI on it with Gradio

`gr.Interface` builds a web page around your function. Tell it the input type and the output type; it draws the rest. `share=True` gives you a temporary public link you can send to anyone.

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classify,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="What's in this picture?",
    description="Upload an image and the model guesses what it sees.",
)

demo.launch(share=True)

The app appears right here in the notebook — drag in a photo and watch the top-3 bars fill in. Scroll up in the output for the `https://....gradio.live` public link; that's the one you can text to a friend. It lives as long as this notebook is running (up to ~72 hours).

## 3. A taste of another modality: make images

Same `from_pretrained` muscle memory, brand-new superpower. This needs the **GPU** on (Runtime → Change runtime type → GPU). `sdxl-turbo` produces a picture in just a couple of steps.

In [ ]:
!pip install -q diffusers accelerate

from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
).to("cuda")

image = pipe("a watercolor fox in a misty forest",
             num_inference_steps=2,
             guidance_scale=0.0).images[0]

image  # in a notebook, this just displays the picture

## 4. Bonus: wrap the image generator in a UI too

The recipe never changed — only the model did. Text in, image out.

In [ ]:
def generate(prompt):
    return pipe(prompt, num_inference_steps=2, guidance_scale=0.0).images[0]

art = gr.Interface(
    fn=generate,
    inputs=gr.Textbox(label="Describe a picture"),
    outputs=gr.Image(label="Result"),
    title="Make a picture from words",
)

art.launch(share=True)

That's the whole series in one notebook: load a pretrained model, wrap it in a function, put a UI on it, share the link. You can now take models off the shelf and run them — and give them a face. Go build something.